# Phase 4: Customer Analysis

This notebook analyzes customer sales, profit, repeat purchases, revenue contribution, retention, and RFM segments.

## 1. Import libraries and load the cleaned dataset

In [1]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()

if current_folder.name == 'notebooks':
    project_folder = current_folder.parent
else:
    project_folder = current_folder

cleaned_file = project_folder / 'data' / 'processed' / 'cleaned_sales.csv'
sales = pd.read_csv(cleaned_file, parse_dates=['Order Date'])

print('Rows loaded:', len(sales))

Rows loaded: 62884


## 2. Calculate customer-wise sales and profit

In [2]:
customer_performance = (
    sales.groupby(['CustomerKey', 'Name'], as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
    )
    .sort_values('Revenue', ascending=False)
)

customer_performance[['Revenue', 'Profit']] = customer_performance[['Revenue', 'Profit']].round(2)
display(customer_performance.head())

,CustomerKey,Name,Revenue,Profit,Orders
9371,1702221,Matthew Flemming,61871.70,39606.02,9
10515,1884663,Karen Jones,43517.80,28819.96,3
11072,1969704,Zrina Topic,42788.04,26008.23,7
2743,535496,Stefanie Hartmann,41521.53,26957.37,3
2840,551036,Stephan Rothstein,40556.54,26716.09,4


## 3. Find repeat customers

In [3]:
repeat_customers = customer_performance[customer_performance['Orders'] > 1].copy()

print('Total customers:', len(customer_performance))
print('Repeat customers:', len(repeat_customers))
display(repeat_customers.head())

Total customers: 11887
Repeat customers: 7272


,CustomerKey,Name,Revenue,Profit,Orders
9371,1702221,Matthew Flemming,61871.70,39606.02,9
10515,1884663,Karen Jones,43517.80,28819.96,3
11072,1969704,Zrina Topic,42788.04,26008.23,7
2743,535496,Stefanie Hartmann,41521.53,26957.37,3
2840,551036,Stephan Rothstein,40556.54,26716.09,4


## 4. Find the top 20 customers

In [4]:
top_20_customers = customer_performance.head(20)
display(top_20_customers)

,CustomerKey,Name,Revenue,Profit,Orders
9371,1702221,Matthew Flemming,61871.70,39606.02,9
10515,1884663,Karen Jones,43517.80,28819.96,3
11072,1969704,Zrina Topic,42788.04,26008.23,7
2743,535496,Stefanie Hartmann,41521.53,26957.37,3
2840,551036,Stephan Rothstein,40556.54,26716.09,4
3664,723572,Gaspare Trevisan,40225.01,23440.78,14
1175,262871,Roy Le,38813.88,23986.19,3
10795,1928466,Dennis Weissmuller,38191.06,24054.60,4
8116,1503831,Virgie Takacs,37319.88,22881.47,3
7275,1373561,Ollie Davis,36817.28,22954.48,5


## 5. Find customers contributing the first 80% of revenue

In [5]:
customer_performance['Cumulative Revenue'] = customer_performance['Revenue'].cumsum()
customer_performance['Cumulative Revenue %'] = (
    customer_performance['Cumulative Revenue']
    / customer_performance['Revenue'].sum()
    * 100
)

previous_cumulative_percent = customer_performance['Cumulative Revenue %'].shift(fill_value=0)
customers_contributing_80_percent = customer_performance[previous_cumulative_percent < 80].copy()

print('Customers contributing the first 80% of revenue:', len(customers_contributing_80_percent))
display(customers_contributing_80_percent)

Customers contributing the first 80% of revenue: 4874


,CustomerKey,Name,Revenue,Profit,Orders,Cumulative Revenue,Cumulative Revenue %
9371,1702221,Matthew Flemming,61871.70,39606.02,9,61871.70,0.110970
10515,1884663,Karen Jones,43517.80,28819.96,3,105389.50,0.189021
11072,1969704,Zrina Topic,42788.04,26008.23,7,148177.54,0.265763
2743,535496,Stefanie Hartmann,41521.53,26957.37,3,189699.07,0.340234
2840,551036,Stephan Rothstein,40556.54,26716.09,4,230255.61,0.412974
...,...,...,...,...,...,...,...
4868,949448,Jacob Ellis,3971.49,2021.11,4,44590965.59,79.975934
3001,580026,Maximilian Meyer,3970.98,2454.64,2,44594936.57,79.983056
11635,2060666,Marvel Roque,3970.96,2243.11,3,44598907.53,79.990178
2329,461824,Niklas Ehrlichmann,3969.00,2143.76,2,44602876.53,79.997297


## 6. Calculate customer retention rate

For this analysis, retention rate is the percentage of customers who placed more than one order.

In [6]:
retention_rate = (len(repeat_customers) / len(customer_performance)) * 100
print(f'Customer retention rate: {retention_rate:.2f}%')

Customer retention rate: 61.18%


## 7. Calculate Recency, Frequency, and Monetary value

Recency is measured from one day after the latest order date in the dataset. Frequency is the number of distinct orders, and Monetary is total customer revenue.

In [7]:
snapshot_date = sales['Order Date'].max() + pd.Timedelta(days=1)

customer_rfm = (
    sales.groupby(['CustomerKey', 'Name'], as_index=False)
    .agg(
        Last_Purchase_Date=('Order Date', 'max'),
        Frequency=('Order Number', 'nunique'),
        Monetary=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
    )
)

customer_rfm['Recency'] = (snapshot_date - customer_rfm['Last_Purchase_Date']).dt.days
customer_rfm[['Monetary', 'Profit']] = customer_rfm[['Monetary', 'Profit']].round(2)

display(customer_rfm.head())

,CustomerKey,Name,Last_Purchase_Date,Frequency,Monetary,Profit,Recency
0,301,Lilly Harding,2019-11-11,1,592.00,395.86,468
1,325,Madison Hull,2020-01-04,3,5787.67,3380.07,414
2,554,Claire Ferres,2019-12-05,2,951.71,504.33,444
3,1042,Aidan Pankhurst,2018-03-06,1,1124.91,732.17,1083
4,1314,Isaac Israel,2017-12-19,1,2539.86,1489.01,1160


## 8. Create RFM scores

Each customer receives a score from 1 to 5 for Recency, Frequency, and Monetary value. A higher score is better.

In [8]:
customer_rfm['R_Score'] = pd.qcut(
    customer_rfm['Recency'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1],
).astype(int)

customer_rfm['F_Score'] = pd.qcut(
    customer_rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

customer_rfm['M_Score'] = pd.qcut(
    customer_rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

customer_rfm['RFM_Score'] = (
    customer_rfm['R_Score']
    + customer_rfm['F_Score']
    + customer_rfm['M_Score']
)

## 9. Create customer segments

In [9]:
def assign_customer_segment(rfm_score):
    if rfm_score >= 13:
        return 'Champions'
    elif rfm_score >= 10:
        return 'Loyal Customers'
    elif rfm_score >= 7:
        return 'Potential Loyalists'
    elif rfm_score >= 4:
        return 'At Risk'
    else:
        return 'Lost Customers'

customer_rfm['Customer Segment'] = customer_rfm['RFM_Score'].apply(assign_customer_segment)

segment_summary = (
    customer_rfm.groupby('Customer Segment', as_index=False)
    .agg(
        Customers=('CustomerKey', 'count'),
        Revenue=('Monetary', 'sum'),
        Profit=('Profit', 'sum'),
    )
    .sort_values('Revenue', ascending=False)
)

segment_summary[['Revenue', 'Profit']] = segment_summary[['Revenue', 'Profit']].round(2)
display(segment_summary)

,Customer Segment,Customers,Revenue,Profit
1,Champions,2163,23189401.59,13679218.65
3,Loyal Customers,3106,18071851.28,10602874.83
4,Potential Loyalists,3463,10783157.65,6284991.23
0,At Risk,2797,3581832.26,2025068.24
2,Lost Customers,358,129236.81,70535.43


## 10. Save the customer segmentation dataset

In [10]:
output_file = project_folder / 'data' / 'processed' / 'customer_segments.csv'
customer_rfm.to_csv(output_file, index=False, date_format='%Y-%m-%d')

print('Customer segmentation dataset saved to:', output_file)

Customer segmentation dataset saved to: d:\Desktop\Pune Internship\Global-Electronics-Retail-Sales-Analysis-and-Business-Intelligence-Dashboard\data\processed\customer_segments.csv


## 11. Display the Phase 4 answers

In [11]:
print('PHASE 4 ANSWERS')
print('-' * 50)
print(f'Total customers: {len(customer_performance):,}')
print(f'Repeat customers: {len(repeat_customers):,}')
print(f'Customer retention rate: {retention_rate:.2f}%')
print(f'Customers contributing the first 80% of revenue: {len(customers_contributing_80_percent):,}')
print(f'Top customer: {top_20_customers.iloc[0]["Name"]}')
print(f'Top customer revenue: ${top_20_customers.iloc[0]["Revenue"]:,.2f}')
display(top_20_customers)
display(segment_summary)

PHASE 4 ANSWERS
--------------------------------------------------
Total customers: 11,887
Repeat customers: 7,272
Customer retention rate: 61.18%
Customers contributing the first 80% of revenue: 4,874
Top customer: Matthew Flemming
Top customer revenue: $61,871.70


,CustomerKey,Name,Revenue,Profit,Orders
9371,1702221,Matthew Flemming,61871.70,39606.02,9
10515,1884663,Karen Jones,43517.80,28819.96,3
11072,1969704,Zrina Topic,42788.04,26008.23,7
2743,535496,Stefanie Hartmann,41521.53,26957.37,3
2840,551036,Stephan Rothstein,40556.54,26716.09,4
3664,723572,Gaspare Trevisan,40225.01,23440.78,14
1175,262871,Roy Le,38813.88,23986.19,3
10795,1928466,Dennis Weissmuller,38191.06,24054.60,4
8116,1503831,Virgie Takacs,37319.88,22881.47,3
7275,1373561,Ollie Davis,36817.28,22954.48,5


,Customer Segment,Customers,Revenue,Profit
1,Champions,2163,23189401.59,13679218.65
3,Loyal Customers,3106,18071851.28,10602874.83
4,Potential Loyalists,3463,10783157.65,6284991.23
0,At Risk,2797,3581832.26,2025068.24
2,Lost Customers,358,129236.81,70535.43
